# Phase 2 Walkthrough — A/B Testing Routing Layer

This notebook walks through the Phase 2 deployment end-to-end:
1. Verify infrastructure
2. Send test requests across all four routing strategies
3. Run CloudWatch Logs Insights queries to analyse traffic, latency, and errors

**Prerequisites:** Run `python infra/phase2_deploy.py` and `python scripts/seed_dynamodb.py` first.

In [ ]:
import json
import time
import boto3
import requests
from pathlib import Path

REPO_ROOT = Path.cwd().parent

with open(REPO_ROOT / 'benchmarks' / 'phase2_outputs.json') as f:
    outputs = json.load(f)

API_URL        = outputs['api_gateway_url']
FUNCTION_NAME  = outputs['lambda_function_name']
DYNAMODB_TABLE = outputs['dynamodb_table']

print('API URL:       ', API_URL)
print('Lambda:        ', FUNCTION_NAME)
print('DynamoDB table:', DYNAMODB_TABLE)

## 1. Verify Infrastructure

In [ ]:
# Lambda function state
lc = boto3.client('lambda', region_name='us-east-1')
fn = lc.get_function(FunctionName=FUNCTION_NAME)['Configuration']
print(f"Function:  {fn['FunctionName']}")
print(f"State:     {fn['State']}")
print(f"Runtime:   {fn['Runtime']}")
print(f"Memory:    {fn['MemorySize']} MB")
print(f"Timeout:   {fn['Timeout']}s")

In [ ]:
# DynamoDB config item
ddb = boto3.resource('dynamodb', region_name='us-east-1')
table = ddb.Table(DYNAMODB_TABLE)
item = table.get_item(Key={'config_id': 'active'})['Item']
print('Active config:')
print(json.dumps(
    {k: float(v) if hasattr(v, '__float__') and not isinstance(v, str)
     else {ik: float(iv) for ik, iv in v.items()} if isinstance(v, dict)
     else v
     for k, v in item.items()},
    indent=2
))

## 2. Smoke Test — Single Request

In [ ]:
resp = requests.post(
    API_URL,
    json={'inputs': 'Why was my card declined?'},
    headers={'Content-Type': 'application/json'},
    timeout=30,
)
print(f'Status: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

## 3. Routing Strategies Demo

In [ ]:
# -- weighted_random: send 20 requests and count variant distribution --
from collections import Counter

results = []
for _ in range(20):
    r = requests.post(API_URL, json={'inputs': 'I lost my card'}, timeout=30)
    results.append(r.json())

counts = Counter(r['variant'] for r in results)
print('Variant distribution (n=20, weights A=0.6 B=0.2 C=0.2):')
for variant, count in sorted(counts.items()):
    print(f'  {variant}: {count} ({count/20*100:.0f}%)')

In [ ]:
# -- header_pinned: force VariantA --
VARIANT_A = 'VariantA-BERT-FP32'
pinned = []
for _ in range(5):
    r = requests.post(
        API_URL,
        json={'inputs': 'My payment failed'},
        headers={'Content-Type': 'application/json', 'X-Target-Variant': VARIANT_A},
        timeout=30,
    )
    pinned.append(r.json()['variant'])

print('header_pinned results (should all be VariantA-BERT-FP32):')
print(pinned)

In [ ]:
# -- switch to least_latency --
from decimal import Decimal

table.update_item(
    Key={'config_id': 'active'},
    UpdateExpression='SET strategy = :s',
    ExpressionAttributeValues={':s': 'least_latency'},
)
print('Switched to least_latency. Waiting 35s for Lambda cache TTL...')
time.sleep(35)

ll_results = [requests.post(API_URL, json={'inputs': 'Check balance'}, timeout=30).json() for _ in range(5)]
print('least_latency variants selected:', [r['variant'] for r in ll_results])

# Restore
table.update_item(
    Key={'config_id': 'active'},
    UpdateExpression='SET strategy = :s',
    ExpressionAttributeValues={':s': 'weighted_random'},
)
time.sleep(35)
print('Restored to weighted_random')

## 4. CloudWatch Logs Insights Queries

Run these queries in the [CloudWatch Logs Insights console](https://console.aws.amazon.com/cloudwatch/home#logsV2:logs-insights) against the log group `/aws/lambda/ab-gateway-router`.

---

### Query 1 — Variant traffic distribution (last 1 hour)

```
fields @timestamp, variant_selected, strategy
| filter ispresent(variant_selected)
| stats count() as requests by variant_selected
| sort requests desc
```

---

### Query 2 — p50/p95 latency per variant

```
fields @timestamp, variant_selected, sagemaker_latency_ms
| filter ispresent(sagemaker_latency_ms)
| stats
    pct(sagemaker_latency_ms, 50) as p50,
    pct(sagemaker_latency_ms, 95) as p95,
    avg(sagemaker_latency_ms) as mean
  by variant_selected
```

---

### Query 3 — Error rate per variant

```
fields @timestamp, variant_selected, error
| stats
    count() as total,
    count(error) as errors
  by variant_selected
| extend error_rate = errors / total * 100
```

---

### Query 4 — Shadow vs primary confidence comparison

```
fields @timestamp, variant_selected, confidence, is_shadow
| filter strategy = "shadow"
| stats avg(confidence) by is_shadow, variant_selected
```

---

### Query 5 — Slowest requests (debugging)

```
fields @timestamp, request_id, variant_selected, total_latency_ms, predicted_label
| filter total_latency_ms > 2000
| sort total_latency_ms desc
| limit 20
```

---

### Query 6 — DynamoDB cache efficiency

```
fields dynamodb_latency_ms
| stats
    count() as total_requests,
    count(dynamodb_latency_ms > 0) as cache_misses,
    avg(dynamodb_latency_ms) as avg_dynamo_ms
```

In [ ]:
# Run Logs Insights queries programmatically
import time as _time

logs = boto3.client('logs', region_name='us-east-1')
LOG_GROUP = '/aws/lambda/ab-gateway-router'

def run_insights_query(query_string: str, minutes_back: int = 60) -> list:
    end   = int(_time.time())
    start = end - minutes_back * 60

    resp = logs.start_query(
        logGroupName=LOG_GROUP,
        startTime=start,
        endTime=end,
        queryString=query_string,
    )
    query_id = resp['queryId']

    while True:
        status = logs.get_query_results(queryId=query_id)
        if status['status'] in ('Complete', 'Failed', 'Cancelled'):
            return status['results']
        _time.sleep(1)


# Query 1 — traffic distribution
q1_results = run_insights_query("""
fields @timestamp, variant_selected, strategy
| filter ispresent(variant_selected)
| stats count() as requests by variant_selected
| sort requests desc
""")

print('Traffic distribution (last 60 min):')
for row in q1_results:
    fields = {f['field']: f['value'] for f in row}
    print(f"  {fields.get('variant_selected', '?'):30s}  {fields.get('requests', '?')} requests")

In [ ]:
# Query 2 — p50/p95 latency per variant
q2_results = run_insights_query("""
fields @timestamp, variant_selected, sagemaker_latency_ms
| filter ispresent(sagemaker_latency_ms)
| stats
    pct(sagemaker_latency_ms, 50) as p50,
    pct(sagemaker_latency_ms, 95) as p95,
    avg(sagemaker_latency_ms) as mean
  by variant_selected
""")

print('SageMaker latency per variant (last 60 min):')
print(f"{'Variant':30s}  {'p50':>8s}  {'p95':>8s}  {'mean':>8s}")
print('-' * 62)
for row in q2_results:
    f = {x['field']: x['value'] for x in row}
    print(f"{f.get('variant_selected','?'):30s}  {f.get('p50','?'):>8s}  {f.get('p95','?'):>8s}  {f.get('mean','?'):>8s}")

In [ ]:
# Query 6 — DynamoDB cache efficiency
q6_results = run_insights_query("""
fields dynamodb_latency_ms
| stats
    count() as total_requests,
    count(dynamodb_latency_ms > 0) as cache_misses,
    avg(dynamodb_latency_ms) as avg_dynamo_ms
""")

print('DynamoDB cache efficiency (last 60 min):')
for row in q6_results:
    f = {x['field']: x['value'] for x in row}
    print(f"  Total requests: {f.get('total_requests','?')}")
    print(f"  Cache misses:   {f.get('cache_misses','?')}")
    print(f"  Avg DynamoDB:   {f.get('avg_dynamo_ms','?')} ms")

## 5. Verification Commands

Run these from the terminal to manually verify the deployment:

```bash
# Lambda function state
aws lambda get-function --function-name ab-gateway-router \
  --query 'Configuration.[FunctionName,State,Runtime,MemorySize]'
# Expected: ["ab-gateway-router", "Active", "python3.12", 512]

# Single request via curl
curl -s -X POST \
  $(python -c "import json; print(json.load(open('benchmarks/phase2_outputs.json'))['api_gateway_url'])") \
  -H 'Content-Type: application/json' \
  -d '{"inputs": "I lost my card"}' | python -m json.tool

# DynamoDB config
aws dynamodb get-item \
  --table-name ab-gateway-routing-config \
  --key '{"config_id": {"S": "active"}}' \
  --query 'Item.strategy'
# Expected: {"S": "weighted_random"}

# EMF metrics (wait ~2 min after first request)
aws cloudwatch get-metric-statistics \
  --namespace ABGateway \
  --metric-name InvocationCount \
  --dimensions Name=Strategy,Value=weighted_random \
  --start-time $(date -u -v-10M +%Y-%m-%dT%H:%M:%SZ) \
  --end-time $(date -u +%Y-%m-%dT%H:%M:%SZ) \
  --period 60 \
  --statistics Sum
```